# Registry Champion and Challenger

## Topic Goal

This notebook properly implements the **Champion / Challenger model registry pattern** in MLflow.

The correct flow is:

1. Train a baseline model and register it as the **champion**.
2. Train a new candidate model and register it as the **challenger**.
3. Use MLflow model aliases to assign:
   - `champion`
   - `challenger`
4. Load models using alias-based URIs.
5. Compare champion and challenger performance.
6. Optionally promote challenger to champion if it performs better.



## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and MLflow client utilities.

`MlflowClient` is important because aliases such as `champion` and `challenger` are managed using the MLflow client API.


In [ ]:
# Import os to create folders and clean MLflow environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking, model logging, and model registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import MlflowClient for managing registry aliases such as champion and challenger.
from mlflow.tracking import MlflowClient

# Import infer_signature for capturing input/output model schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing tabular data.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import train_test_split for splitting dataset.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for applying different transformations to different column types.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical variables.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical variables.
from sklearn.preprocessing import StandardScaler

# Import Pipeline for combining preprocessing and model.
from sklearn.pipeline import Pipeline

# Import models for champion and challenger demonstration.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow

This notebook uses a local `mlruns` folder for easy Windows classroom demonstration.

The registered model name is created from the experiment name.


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_07_registry_champion_challenger"

# Define champion run name.
CHAMPION_RUN_NAME = "07_registry_champion_baseline_model"

# Define challenger run name.
CHALLENGER_RUN_NAME = "07_registry_challenger_candidate_model"

# Remove inherited MLflow Project environment variables if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for generated files.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for additional files.
os.makedirs("artifacts", exist_ok=True)

# Set local MLflow tracking URI.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Create MLflow client for registry operations.
client = MlflowClient()

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load and Prepare Dataset

We use the customer churn dataset.

The model predicts whether a customer will churn.


In [ ]:
# Load customer churn dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split dataset into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print dataset information.
print("Dataset shape:", df.shape)
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 4. Helper Function for Training, Logging, and Registering a Model

This helper function keeps the notebook clean.

It performs these steps in the **same MLflow run**:

1. Train the model.
2. Evaluate the model.
3. Infer signature.
4. Log parameters and metrics.
5. Log the model with signature and input example.
6. Create `model_uri` from the same active run.
7. Register the model using that same `model_uri`.

This avoids the earlier mistake of creating a separate run only for signature and registry.


In [ ]:
# Define a reusable helper function.
def train_log_and_register_model(run_name, model_label, estimator):
    # Create a full sklearn pipeline.
    pipeline = Pipeline(steps=[
        # Apply preprocessing first.
        ("preprocessor", preprocessor),

        # Apply estimator after preprocessing.
        ("model", estimator),
    ])

    # Train the pipeline.
    pipeline.fit(X_train, y_train)

    # Predict on test data.
    predictions = pipeline.predict(X_test)

    # Calculate accuracy.
    accuracy = accuracy_score(y_test, predictions)

    # Calculate precision.
    precision = precision_score(y_test, predictions, zero_division=0)

    # Calculate recall.
    recall = recall_score(y_test, predictions, zero_division=0)

    # Calculate F1-score.
    f1 = f1_score(y_test, predictions, zero_division=0)

    # Create input example for signature.
    input_example = X_test.head(5)

    # Generate sample predictions for signature.
    sample_predictions = pipeline.predict(input_example)

    # Infer model signature.
    signature = infer_signature(input_example, sample_predictions)

    # Create classification report.
    report = classification_report(y_test, predictions, zero_division=0)

    # Save classification report locally.
    report_path = f"outputs/{model_label}_classification_report.txt"
    with open(report_path, "w") as file:
        file.write(report)

    # Start the SAME run for metrics, model, signature, model URI, and registry source.
    with mlflow.start_run(run_name=run_name):

        # Log dataset path.
        mlflow.log_param("dataset", DATA_PATH)

        # Log model label such as champion_candidate or challenger_candidate.
        mlflow.log_param("model_label", model_label)

        # Log model class name.
        mlflow.log_param("model_class", estimator.__class__.__name__)

        # Log accuracy.
        mlflow.log_metric("accuracy", accuracy)

        # Log precision.
        mlflow.log_metric("precision", precision)

        # Log recall.
        mlflow.log_metric("recall", recall)

        # Log F1-score.
        mlflow.log_metric("f1_score", f1)

        # Log classification report artifact.
        mlflow.log_artifact(report_path)

        # Log model with signature and input example in the SAME run.
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            input_example=input_example,
            signature=signature
        )

        # Create model URI from the same active run.
        model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

    # Register the model using the same-run model URI.
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=REGISTERED_MODEL_NAME
    )

    # Return useful details.
    return {
        "pipeline": pipeline,
        "model_uri": model_uri,
        "registered_model": registered_model,
        "version": registered_model.version,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }


## 5. Train and Register Baseline Model as Champion

In a real project, the champion is usually the currently approved model.

Here, we train a simple baseline model using Logistic Regression and assign it the alias:

```text
champion
```


In [ ]:
# Create baseline Logistic Regression model.
champion_estimator = LogisticRegression(max_iter=1000)

# Train, log, and register the baseline model.
champion_result = train_log_and_register_model(
    run_name=CHAMPION_RUN_NAME,
    model_label="champion_candidate",
    estimator=champion_estimator
)

# Assign alias champion to the registered version.
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="champion",
    version=champion_result["version"]
)

# Print champion details.
print("Champion model version:", champion_result["version"])
print("Champion F1-score:", champion_result["f1_score"])
print("Champion model URI:", champion_result["model_uri"])
print("Champion alias assigned: champion")


## 6. Train and Register New Candidate Model as Challenger

The challenger is the new model candidate that may replace the champion.

Here, we train a Random Forest model and assign it the alias:

```text
challenger
```


In [ ]:
# Create challenger Random Forest model.
challenger_estimator = RandomForestClassifier(
    n_estimators=180,
    max_depth=8,
    random_state=42
)

# Train, log, and register the challenger model.
challenger_result = train_log_and_register_model(
    run_name=CHALLENGER_RUN_NAME,
    model_label="challenger_candidate",
    estimator=challenger_estimator
)

# Assign alias challenger to the registered version.
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="challenger",
    version=challenger_result["version"]
)

# Print challenger details.
print("Challenger model version:", challenger_result["version"])
print("Challenger F1-score:", challenger_result["f1_score"])
print("Challenger model URI:", challenger_result["model_uri"])
print("Challenger alias assigned: challenger")


## 7. Load Models Using Aliases

Alias-based model loading is better than hardcoding model versions.

Examples:

```text
models:/ModelName@champion
models:/ModelName@challenger
```

This is useful because deployment code can refer to aliases instead of changing version numbers manually.


In [ ]:
# Create champion model URI using alias.
champion_alias_uri = f"models:/{REGISTERED_MODEL_NAME}@champion"

# Create challenger model URI using alias.
challenger_alias_uri = f"models:/{REGISTERED_MODEL_NAME}@challenger"

# Print alias URIs.
print("Champion alias URI:", champion_alias_uri)
print("Challenger alias URI:", challenger_alias_uri)

# Load champion model using alias.
champion_model = mlflow.pyfunc.load_model(champion_alias_uri)

# Load challenger model using alias.
challenger_model = mlflow.pyfunc.load_model(challenger_alias_uri)

# Create sample input.
sample_input = X_test.head(5)

# Predict using champion.
champion_predictions = champion_model.predict(sample_input)

# Predict using challenger.
challenger_predictions = challenger_model.predict(sample_input)

# Show predictions side by side.
comparison_df = sample_input.copy()
comparison_df["champion_prediction"] = champion_predictions
comparison_df["challenger_prediction"] = challenger_predictions

# Display comparison.
display(comparison_df)


## 8. Compare Champion and Challenger

Now we compare model performance.

If the challenger performs better than the champion, we can promote the challenger to champion.


In [ ]:
# Create comparison table.
model_comparison = pd.DataFrame([
    {
        "alias": "champion",
        "version": champion_result["version"],
        "model_type": "LogisticRegression",
        "accuracy": champion_result["accuracy"],
        "precision": champion_result["precision"],
        "recall": champion_result["recall"],
        "f1_score": champion_result["f1_score"],
    },
    {
        "alias": "challenger",
        "version": challenger_result["version"],
        "model_type": "RandomForestClassifier",
        "accuracy": challenger_result["accuracy"],
        "precision": challenger_result["precision"],
        "recall": challenger_result["recall"],
        "f1_score": challenger_result["f1_score"],
    },
])

# Save comparison table.
model_comparison.to_csv("outputs/champion_challenger_comparison.csv", index=False)

# Display comparison table.
display(model_comparison)

# Check whether challenger is better.
challenger_is_better = challenger_result["f1_score"] > champion_result["f1_score"]

# Print decision.
if challenger_is_better:
    print("Challenger performs better than champion.")
else:
    print("Champion remains better or equal.")


## 9. Optional Promotion: Promote Challenger to Champion

This section demonstrates realistic production behavior.

If the challenger model performs better, we update the `champion` alias to point to the challenger version.

The old champion version remains registered, but the alias now points to the new version.


In [ ]:
# Promote challenger only if it performs better.
if challenger_is_better:
    # Reassign champion alias to challenger version.
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="champion",
        version=challenger_result["version"]
    )

    # Keep challenger alias also pointing to the same version for demonstration.
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="challenger",
        version=challenger_result["version"]
    )

    # Print promotion message.
    print("Challenger promoted to champion.")
    print("New champion version:", challenger_result["version"])

else:
    # Keep current champion unchanged.
    print("No promotion done.")
    print("Current champion version remains:", champion_result["version"])




- A **champion** is the currently approved or production-ready model.
- A **challenger** is a new candidate model being evaluated.
- MLflow aliases allow us to refer to models using names like `champion` and `challenger`.
- This is better than hardcoding model version numbers.
- Each model version is still created from the same run that logged its metrics, signature, input example, and model artifact.
- The champion/challenger decision should normally depend on business-approved metrics such as recall, F1-score, latency, or cost.
- In this notebook, we use F1-score for the promotion decision.
